## Load dataset


In [1]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

In [2]:
from tqdm.notebook import tqdm

In [3]:
import os
import json

In [4]:
base_path = "/home/bruno/storage/data/fma/fma_large/pt_rock_electronic"
dataset_id = "musicnn"

dataset_path = os.path.join(base_path, dataset_id)

In [5]:
BUFFER_SIZE = 10

class HMCDatasetFeatures(Dataset):
    def __init__(self, files):
        self.files = [os.path.join(files, f) for f in os.listdir(files)]  # Lista de arquivos
        self.files.sort()  # Ordena para manter a ordem consistente
        self.buffer = []  # Buffer para armazenar dados carregados temporariamente
        self.buffer_index = 0  # Índice do buffer
    
    def load_file(self, file_path):
        """ Carrega um único arquivo e move para GPU """
        loaded_data = torch.load(file_path, map_location="cpu")  # Carregar na CPU primeiro
        for example in loaded_data:
            example["features"] = torch.tensor(example["features"], dtype=torch.float32, device="cuda")
        return loaded_data

    def __len__(self):
        return sum(len(torch.load(f, map_location="cpu")) for f in self.files)  # Tamanho total

    def __getitem__(self, idx):
        """ Retorna um item específico, carregando arquivos sob demanda """
        if self.buffer_index <= idx < self.buffer_index + len(self.buffer):
            # Se o índice estiver no buffer, apenas retorna
            example = self.buffer[idx - self.buffer_index]
        else:
            # Carregar novo buffer
            file_index = idx // BUFFER_SIZE  # Descobre qual arquivo carregar
            self.buffer = self.load_file(self.files[file_index])  # Carrega o arquivo
            self.buffer_index = file_index * BUFFER_SIZE  # Atualiza índice base
            example = self.buffer[idx - self.buffer_index]  # Pega o exemplo correto
        
        return example["track_id"], example["features"]
    def to_dataframe(self):
        """ Converte o dataset inteiro para um DataFrame Pandas """
        all_data = []
        for file in tqdm(self.files, desc="Carregando arquivos"):
            loaded_data = self.load_file(file)  # Carrega os dados do arquivo
            for example in loaded_data:
                track_id = example["track_id"]
                features = example["features"].cpu().numpy()  # Converte tensor para numpy
                all_data.append({"track_id": track_id, "features": features})

        df = pd.DataFrame(all_data)
        return df


In [6]:
import sys
import types

# Create a dummy dataset module with dataset_torch submodule
dataset = types.ModuleType('dataset')
dataset_torch = types.ModuleType('dataset_torch')
dataset.dataset_torch = dataset_torch
sys.modules['dataset'] = dataset
sys.modules['dataset.dataset_torch'] = dataset_torch


dataset_torch.MusicDataset = HMCDatasetFeatures
sys.modules['dataset_torch'] = dataset_torch


In [7]:
# Carregar o dataset salvo
loaded_dataset = HMCDatasetFeatures(dataset_path)

In [8]:
df = loaded_dataset.to_dataframe()

Carregando arquivos:   0%|          | 0/23 [00:00<?, ?it/s]

In [9]:
df

,track_id,features
0,135,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,136,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,151,"[1.940759, 2.4487689, 2.8015103, 2.9974098, 3...."
3,152,"[1.4773003, 1.3780526, 1.0749683, 0.71745515, ..."
4,153,"[1.5350242, 2.0104704, 2.3191102, 2.4830873, 2..."
...,...,...
23525,155315,"[4.0290804, 4.2446947, 4.3559194, 4.3897634, 4..."
23526,155316,"[3.0930243, 3.5147276, 3.7395504, 4.0232067, 4..."
23527,155317,"[3.5839074, 4.1035447, 4.321473, 4.4095073, 4...."
23528,155318,"[1.7701645, 3.3962436, 4.0411396, 4.360858, 4...."


In [41]:
track_id, features = loaded_dataset[10]

In [42]:
np_features = features.cpu().numpy()

In [43]:
print(track_id)  # Deve ser um ID válido
print(np_features.shape)  # Deve mostrar "cuda:0"

172
(180192,)


In [45]:
np_features[97]

np.float32(4.921104)

In [ ]:
# Criação do DataLoader
ds = DataLoader(loaded_dataset, batch_size=64, shuffle=True, num_workers=4)


/tmp/ipykernel_95186/2141262964.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return sum(len(torch.load(f, map_location="cpu")) for f in self.files)  # Tamanho total


In [ ]:
len(loaded_dataset)

In [25]:
for track_id, feature in ds_validation:
    print(track_id)
    break

tensor([[0., 1.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [1

In [18]:
#ds_train = Dataset(trainset_pattern, 10, 32, depth=metadata['max_depth']).build(df=True)

In [28]:
val_dataset_array = next(iter(ds_validation))[0].numpy()

In [29]:
val_dataset_array

array([[-0.05795394, -0.03638636,  0.22502355, ...,  0.01868322,
         0.16242169,  0.0790742 ],
       [-0.01704843,  0.01406341,  0.00475705, ..., -0.00316535,
         0.19925644,  0.01645163],
       [-0.01152799, -0.05991222, -0.03120563, ...,  0.0007601 ,
        -0.04211006, -0.02922575],
       ...,
       [ 0.00217863,  0.01848902, -0.03141269, ...,  0.00428359,
         0.07604198, -0.00207548],
       [ 0.50329274,  0.05604462,  0.04162657, ..., -0.01690188,
        -0.01881945, -0.04627268],
       [ 0.05001318, -0.03526527,  0.02279306, ...,  0.01409413,
        -0.01773393,  0.00630476]], dtype=float32)